# Beginner Insider Threat Detection with Logon Data

This notebook is a **super beginner-friendly cybersecurity mini-project** using the CERT r4.2 `logon.csv` file.

We will look for **users who log in at unusual times**:
- **After-hours** (before 8am or after 6pm)
- **Weekends** (Saturday/Sunday)

Then we’ll use a simple machine learning model (**Isolation Forest**) to flag suspicious **user-days**.


In [ ]:
# Step 1: Import the libraries we need
# This step is important because Python won't know these tools unless we import them first.

import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest

import matplotlib.pyplot as plt
import seaborn as sns

# Make plots look a bit nicer
sns.set_style("whitegrid")

print("✅ Libraries imported successfully!")


In [ ]:
# Step 2: Load the logon.csv file
# Beginners often forget to update the file path. If you get "FileNotFoundError",
# it usually means the path is wrong.

# IMPORTANT: Change this path to where YOUR logon.csv is located.
file_path = "logon.csv"  # Example: "/home/yourname/Downloads/logon.csv"

try:
    df = pd.read_csv(file_path)
    print("✅ File loaded successfully!")
except FileNotFoundError:
    print("❌ File not found. Fix the path in file_path.")
    print("   Example paths:")
    print("   - Windows: r'C:\\Users\\YourName\\Downloads\\logon.csv'")
    print("   - Linux/Mac: '/home/yourname/Downloads/logon.csv'")
    raise

print("\n📌 First 5 rows:")
print(df.head())

print("\n📌 Columns:")
print(df.columns)

print("\n📌 Shape (rows, columns):")
print(df.shape)


In [ ]:
# Step 3: Convert the 'time' column to datetime
# This step is important because dates/times in CSV are usually strings.
# We need real datetime objects to extract hour, weekday, etc.

# Common beginner mistake: the column might not be named exactly 'time'.
# If this errors, check df.columns above.
df["time"] = pd.to_datetime(df["time"], errors="coerce")

# If some rows failed to convert, they become NaT (missing datetime)
bad_times = df["time"].isna().sum()
print(f"⚠️ Rows with invalid/missing time after conversion: {bad_times}")

# Beginners often forget: handle missing values so later steps don't crash
df = df.dropna(subset=["time"]).copy()
print(f"✅ Rows after dropping invalid times: {len(df)}")

# Extract simple time-based features (beginner-friendly)
df["date"] = df["time"].dt.date
df["hour"] = df["time"].dt.hour
df["weekday"] = df["time"].dt.weekday  # Monday=0 ... Sunday=6

print("\n📌 Preview of time features:")
print(df[["time", "date", "hour", "weekday"]].head())


In [ ]:
# Step 4: Create a simple flag for after-hours OR weekend logins
# Our rule:
# - after-hours if hour < 8 OR hour > 18
# - OR weekend if weekday >= 5 (Saturday=5, Sunday=6)
#
# This step is important because it turns 'time' into a simple yes/no signal.

df["is_after_hours"] = ((df["hour"] < 8) | (df["hour"] > 18) | (df["weekday"] >= 5)).astype(int)

print("📌 is_after_hours value counts (0=normal, 1=after-hours/weekend):")
print(df["is_after_hours"].value_counts(dropna=False))


In [ ]:
# Step 5: Group by user + date
# We want one row per (user, day) so we can spot unusual daily patterns.

# In CERT logon.csv, the user column is usually named 'user'.
# If your file uses a different name, change it here.
user_col = "user"

if user_col not in df.columns:
    print("❌ Could not find the 'user' column.")
    print("   Available columns are:")
    print(df.columns)
    raise KeyError("Please set user_col to the correct column name for the username.")

daily = df.groupby([user_col, "date"]).agg(
    logins_total=("time", "count"),
    after_hours_logins=("is_after_hours", "sum"),
).reset_index()

print("✅ Daily table created!")
print("\n📌 First 10 user-days:")
print(daily.head(10))

print("\n📌 Summary stats:")
print(daily[["logins_total", "after_hours_logins"]].describe())


In [ ]:
# Step 6: Create after_hours_ratio
# This ratio helps normalize behavior:
# - 2 after-hours logins might be suspicious if total logins = 2 (ratio=1.0)
# - but less suspicious if total logins = 100 (ratio=0.02)
#
# Beginners often hit division-by-zero issues, so we protect against that.

# Replace any missing totals (shouldn't happen, but it's beginner-safe)
daily["logins_total"] = daily["logins_total"].fillna(0)
daily["after_hours_logins"] = daily["after_hours_logins"].fillna(0)

# Avoid division by zero using np.where
daily["after_hours_ratio"] = np.where(
    daily["logins_total"] > 0,
    daily["after_hours_logins"] / daily["logins_total"],
    0.0
)

print("📌 First 10 rows with ratio:")
print(daily[[user_col, "date", "logins_total", "after_hours_logins", "after_hours_ratio"]].head(10))


In [ ]:
# Step 7: Train Isolation Forest
# Isolation Forest is an 'anomaly detection' model.
# It tries to find unusual points compared to the rest of the dataset.
#
# We keep it simple: only TWO features:
# - logins_total
# - after_hours_ratio

features = ["logins_total", "after_hours_ratio"]
X = daily[features].copy()

# Beginner-friendly: fill missing values (just in case)
X = X.fillna(0)

model = IsolationForest(
    n_estimators=200,
    contamination=0.01,     # 1% of days expected to be anomalies (you can change this)
    random_state=42
)

model.fit(X)

# Step 8: Add anomaly label
# In sklearn IsolationForest:
# -1 = anomaly (suspicious)
#  1 = normal
daily["anomaly"] = model.predict(X)

# Simple risk_score: 100 if anomaly else 10
daily["risk_score"] = np.where(daily["anomaly"] == -1, 100, 10)

print("✅ Model trained and predictions added!")
print("\n📌 Anomaly counts:")
print(daily["anomaly"].value_counts())


In [ ]:
# Step 9: Print results so beginners immediately see what happened
# We'll show:
# (A) Top suspicious user-days
# (B) Most suspicious users overall (by count of suspicious days)

suspicious_days = daily[daily["anomaly"] == -1].copy()

print(f"✅ Total suspicious user-days found: {len(suspicious_days)}")

# Show top suspicious days (sorted by ratio first, then total logins)
top_days = suspicious_days.sort_values(
    by=["after_hours_ratio", "after_hours_logins", "logins_total"],
    ascending=False
).head(15)

print("\n🚩 Top suspicious user-days (up to 15):")
print(top_days[[user_col, "date", "logins_total", "after_hours_logins", "after_hours_ratio", "risk_score"]])

# Most suspicious users overall
# Count how many suspicious days each user has
user_summary = suspicious_days.groupby(user_col).agg(
    suspicious_days=("date", "count"),
    avg_after_hours_ratio=("after_hours_ratio", "mean"),
    total_after_hours_logins=("after_hours_logins", "sum"),
).reset_index()

user_summary = user_summary.sort_values(
    by=["suspicious_days", "avg_after_hours_ratio"],
    ascending=False
)

print("\n👤 Most suspicious users overall (top 15):")
print(user_summary.head(15))


In [ ]:
# Step 10 (GUI): Pick a user from a dropdown and plot after_hours_ratio over time
# This is a simple "GUI" inside Jupyter using ipywidgets.
# Beginners often like this because you can explore users without editing code each time.

# Try importing ipywidgets. If it's not installed, we show a helpful message.
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    widgets_ok = True
except ImportError:
    widgets_ok = False

if not widgets_ok:
    print("❌ ipywidgets is not installed, so the dropdown GUI cannot show.")
    print("✅ Fix (choose ONE):")
    print("   - If you use pip:   pip install ipywidgets")
    print("   - If you use conda: conda install -c conda-forge ipywidgets")
    print("\nAfter installing, restart the kernel and run this cell again.")
else:
    # Build a list of users for the dropdown.
    # We keep it simple: show all users (may be a lot). If it's too slow, we can limit later.
    all_users = sorted(daily[user_col].dropna().unique().tolist())

    # Helpful default: pick the most suspicious user if we have anomalies.
    if "suspicious_days" in globals() and len(suspicious_days) > 0:
        default_user = suspicious_days.sort_values(
            by=["after_hours_ratio", "after_hours_logins"],
            ascending=False
        ).iloc[0][user_col]
        print(f"✅ Default dropdown user (most suspicious found): {default_user}")
    else:
        default_user = all_users[0]
        print(f"ℹ️ Default dropdown user: {default_user}")
        print("Tip: If you expected anomalies, try contamination=0.02 or 0.05.")

    user_dropdown = widgets.Dropdown(
        options=all_users,
        value=default_user,
        description="User:",
        disabled=False
    )

    out = widgets.Output()

    def plot_for_user(selected_user):
        # This function re-draws the plot each time you pick a new user.
        # clear_output keeps the plot area clean.
        with out:
            clear_output(wait=True)

            one = daily[daily[user_col] == selected_user].copy()
            one["date"] = pd.to_datetime(one["date"], errors="coerce")
            one = one.dropna(subset=["date"]).sort_values("date")

            # Beginner-safe: if no rows, show a message instead of crashing.
            if len(one) == 0:
                print("No data found for that user.")
                return

            avg_ratio = one["after_hours_ratio"].mean()

            print(f"Showing user: {selected_user}")
            print("\n📌 First 5 days:")
            print(one[[user_col, "date", "logins_total", "after_hours_ratio", "anomaly"]].head(5))

            plt.figure(figsize=(12, 5))
            plt.plot(one["date"], one["after_hours_ratio"], marker="o", linewidth=2, label="after_hours_ratio")
            plt.axhline(avg_ratio, color="red", linestyle="--", linewidth=2, label=f"Average = {avg_ratio:.2f}")
            plt.title(f"After-Hours Login Ratio Over Time for User: {selected_user}")
            plt.xlabel("Date")
            plt.ylabel("After-Hours Ratio (0 to 1)")
            plt.legend()
            plt.tight_layout()
            plt.show()

    # Connect dropdown changes to the plot function.
    def on_change(change):
        if change["name"] == "value":
            plot_for_user(change["new"])

    user_dropdown.observe(on_change)

    # Display the GUI widgets and draw the first plot.
    display(user_dropdown)
    display(out)

    plot_for_user(user_dropdown.value)


## What You Learned + Next Steps

In this beginner project, you learned how to:
- Load a real cybersecurity dataset (`logon.csv`) with `pandas`
- Convert a text time column into `datetime`
- Create a simple rule-based signal: **after-hours/weekend logins**
- Aggregate raw logs into **daily user behavior**
- Use a basic anomaly detection model (**Isolation Forest**) to flag unusual user-days
- Print actionable results and visualize a user’s behavior trend

### Next Steps (keep it simple)
- Add **device context** later by joining with `device.csv` (example idea: “after-hours logins from unusual devices”)
- Try different `contamination` values like `0.02` or `0.05` to see more/fewer alerts
- Add a very basic rule-based alert too (example: `after_hours_ratio > 0.8`)
